# 10. Compile the paper and the supplement (optional)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/revision/v40.31-scientific/colab/10_compile_paper.ipynb)

Installs TeX Live (about 5 minutes), compiles both documents with pdflatex, the engine the MATCH template names, and runs the label and cross-reference check. The PDFs are copied to the state folder. Their bytes depend on the TeX installation, so they are not compared with the committed PDFs.

In [ ]:
# Setup (the same cell in every notebook of colab/; about 2 minutes the first time in a session)
REVISION = "revision/v40.31-scientific"   # the branch or tag to run; every notebook of one run must use the same commit
REPO_URL = "https://github.com/Singati2/LOYOLA-PAPER.git"
import os, sys, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")             # progress and outputs are kept in MyDrive/LOYOLA_colab_run
    WORK = "/content/LOYOLA-PAPER"
    if not os.path.exists(WORK):
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")  # Colab secret (key icon on the left), needed while the repository is private
        except Exception:
            pass
        bundle = "/content/drive/MyDrive/LOYOLA-PAPER.bundle"
        if token:
            url = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
            subprocess.run(["git", "clone", "--quiet", "--branch", REVISION, url, WORK], check=True)
            subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", REPO_URL], check=True)   # token not kept on disk
        elif os.path.exists(bundle):
            subprocess.run(["git", "clone", "--quiet", "--branch", REVISION, bundle, WORK], check=True)
        else:
            subprocess.run(["git", "clone", "--quiet", "--branch", REVISION, REPO_URL, WORK], check=True)
    os.chdir(WORK)
    ENV = "/content/loyola-env"
else:                                         # run locally from inside the repository
    while not os.path.exists("verify.py") and os.getcwd() != "/":
        os.chdir("..")
    ENV = os.path.join(os.getcwd(), ".venv-colab")
PY = os.path.join(ENV, "bin", "python")
if not os.path.exists(PY):                    # Python 3.13 and the pinned versions that produced the committed outputs
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
    UV = ["uv"] if shutil.which("uv") else [sys.executable, "-m", "uv"]
    subprocess.run(UV + ["venv", "--quiet", "--python", "3.13", ENV], check=True)
    subprocess.run(UV + ["pip", "install", "--quiet", "--python", PY, "-r", "requirements-lock.txt",
                         "-r", "requirements-extras-lock.txt"], check=True)
print(subprocess.run([PY, "-c", "import sys, numpy, scipy; print('python', sys.version.split()[0], '| numpy', "
                      "numpy.__version__, '| scipy', scipy.__version__)"], capture_output=True, text=True).stdout.strip())
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
STATE = os.environ.get("LOYOLA_STATE") or ("/content/drive/MyDrive/LOYOLA_colab_run" if IN_COLAB
                                           else os.path.join(os.getcwd(), "colab_run"))   # as in pipeline.py


def pipeline(*args):
    """Run colab/pipeline.py with the pinned Python, show its output as it comes, stop on failure."""
    p = subprocess.Popen([PY, "-u", "colab/pipeline.py", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("pipeline.py " + " ".join(args) + " reported a problem (see the output above)")


In [ ]:
subprocess.run("apt-get -qq update && apt-get -qq install -y texlive-latex-recommended texlive-latex-extra "
               "texlive-fonts-recommended lmodern > /dev/null", shell=True, check=True)
import tempfile
build = tempfile.mkdtemp()
for f in ("main.tex", "supplement.tex"):
    shutil.copy(f, build)
shutil.copytree("figures", os.path.join(build, "figures"))
for doc in ("main", "supplement"):
    for _ in range(3):
        r = subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", doc + ".tex"], cwd=build,
                           capture_output=True, text=True, errors="replace")   # TeX logs are not UTF-8
    print(doc, "exit", r.returncode, "|", r.stdout.strip().splitlines()[-1])
    if r.returncode != 0:
        raise RuntimeError(doc + ".tex did not compile")
state = STATE
os.makedirs(os.path.join(state, "paper"), exist_ok=True)
for doc, out in (("main", "LOYOLA_paper.pdf"), ("supplement", "LOYOLA_supplement.pdf")):
    shutil.copy(os.path.join(build, doc + ".pdf"), os.path.join(state, "paper", out))
r = subprocess.run([PY, "audit/check_labels.py"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
if r.returncode != 0:
    raise RuntimeError("label and reference check failed")